# Assignment — Bloc 6
### Mini-repte: extracció de features espectrals

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Colab el carregarà directament des de GitHub. Per guardar els teus canvis: `File → Save a copy in Drive`.

Avui completaràs dues funcions curtes que, juntes, construeixen el dataset de features que farem servir a la **Sessió 12** per entrenar un classificador.

Completa les funcions marcades amb `# TODO`. Cada una té el seu autotest: si l'autotest passa (✅), aquella part ja és correcta.

In [ ]:
import numpy as np
import librosa
import glob
import warnings
warnings.filterwarnings('ignore')

# Clona el repositori per accedir al mini-dataset
import os
if not os.path.exists('/content/tp2'):
    os.system('git clone -q https://github.com/jcomajuncosas/tp2.git /content/tp2')
DATASET_DIR = "/content/tp2/06_analisi_ml/sessio11_fft_features/dataset"

# Si el clone no funciona (repositori privat), puja manualment la carpeta
# 'dataset/' i ajusta aquesta ruta:
# DATASET_DIR = "/content/dataset"

print("Dataset disponible:", os.path.exists(DATASET_DIR))

## TODO 1 — `freq_pic(audio, sr)`

Completa una funció que, donat un array d'àudio i el seu sample rate, retorni la freqüència (en Hz) on hi ha el pic d'energia de l'espectre — fent servir `np.fft.rfft()`, igual que a `patches_bloc6.ipynb` Secció 1.

In [ ]:
def freq_pic(audio, sr):
    """
    Retorna la freqüència (Hz) on l'espectre d'amplitud de 'audio' té
    el seu valor màxim.

    Patró (igual que a patches_bloc6.ipynb):
      1. calcula la FFT amb np.fft.rfft(audio)
      2. calcula les freqüències corresponents amb np.fft.rfftfreq(len(audio), 1/sr)
      3. calcula la magnitud (valor absolut) de la FFT
      4. retorna la freqüència on la magnitud és màxima
    """
    # TODO: implementa els 4 passos de la docstring
    fft_result = None       # <-- substitueix
    freqs = None             # <-- substitueix
    magnitude = None         # <-- substitueix
    return None               # <-- substitueix (retorna la freq del pic)

In [ ]:
def _test_freq_pic():
    print("Test 1: freq_pic()...")

    # Un to pur de 440Hz hauria de donar un pic molt a prop de 440Hz
    sr = 16000
    duration = 0.5
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    senyal_440 = np.sin(2 * np.pi * 440 * t)

    resultat = freq_pic(senyal_440, sr)
    assert resultat is not None, "❌ freq_pic() no retorna res"
    assert abs(resultat - 440) < 5, f"❌ Esperava ~440Hz, ha donat {resultat}Hz"

    # Un to de 2000Hz hauria de donar un pic a prop de 2000Hz
    senyal_2000 = np.sin(2 * np.pi * 2000 * t)
    resultat2 = freq_pic(senyal_2000, sr)
    assert abs(resultat2 - 2000) < 5, f"❌ Esperava ~2000Hz, ha donat {resultat2}Hz"

    print("✅ freq_pic() correcte\n")

_test_freq_pic()

## TODO 2 — `extreu_features(filepath)`

Completa una funció que, donat el camí a un fitxer WAV, en carregui l'àudio amb `librosa.load()` i en calculi tres features: **centroide espectral**, **ZCR** i **MFCC** (13 coeficients), tal com vam veure a `patches_bloc6.ipynb` Secció 2.

In [ ]:
def extreu_features(filepath):
    """
    Carrega l'àudio de 'filepath' i en retorna un diccionari amb:
      - 'centroid': centroide espectral mitjà (librosa.feature.spectral_centroid)
      - 'zcr': zero crossing rate mitjà (librosa.feature.zero_crossing_rate)
      - 'mfcc_0' ... 'mfcc_12': els 13 coeficients MFCC mitjans
        (librosa.feature.mfcc amb n_mfcc=13)

    Pista: cada feature de librosa retorna un array 2D; cal fer .mean()
    (o .mean(axis=1) per al MFCC, que té 13 files) per obtenir un sol
    valor (o un valor per coeficient).

    Utilitza n_fft = min(2048, len(audio)) per evitar avisos en sons curts.
    """
    audio, sr = librosa.load(filepath, sr=None)
    n_fft = min(2048, len(audio))

    # TODO: calcula centroid, zcr i mfcc fent servir librosa
    centroid = None  # <-- substitueix
    zcr = None        # <-- substitueix
    mfcc = None        # <-- substitueix (array de 13 valors, un per coeficient)

    result = {'centroid': centroid, 'zcr': zcr}
    for i, val in enumerate(mfcc):
        result[f'mfcc_{i}'] = val
    return result

In [ ]:
def _test_extreu_features():
    print("Test 2: extreu_features()...")

    kick_files = sorted(glob.glob(f"{DATASET_DIR}/kick/*.wav"))
    hihat_files = sorted(glob.glob(f"{DATASET_DIR}/hihat/*.wav"))

    assert len(kick_files) > 0, "❌ No s'han trobat fitxers de kick — revisa DATASET_DIR"
    assert len(hihat_files) > 0, "❌ No s'han trobat fitxers de hihat — revisa DATASET_DIR"

    feat_kick = extreu_features(kick_files[0])
    feat_hihat = extreu_features(hihat_files[0])

    assert feat_kick is not None, "❌ extreu_features() no retorna res"
    assert 'centroid' in feat_kick and feat_kick['centroid'] is not None, "❌ falta 'centroid'"
    assert 'zcr' in feat_kick and feat_kick['zcr'] is not None, "❌ falta 'zcr'"
    assert all(f'mfcc_{i}' in feat_kick for i in range(13)), "❌ falten coeficients MFCC (n'hi ha d'haver 13)"
    assert all(feat_kick[f'mfcc_{i}'] is not None for i in range(13)), "❌ algun MFCC és None"

    # Comprovem que el kick i el hihat es distingeixen pel centroide
    assert feat_kick['centroid'] < feat_hihat['centroid'], (
        "❌ El centroide del kick hauria de ser més baix que el del hihat "
        "(el kick concentra energia en freqüències baixes)"
    )

    print(f"   Centroid kick: {feat_kick['centroid']:.0f}Hz, hihat: {feat_hihat['centroid']:.0f}Hz")
    print("✅ extreu_features() correcte\n")

_test_extreu_features()

## Construint el dataset complet

Amb les dues funcions ja fetes, construïm el CSV final amb totes les mostres del mini-dataset (no cal tocar res d'aquí en avall).

In [ ]:
import pandas as pd

kick_files = sorted(glob.glob(f"{DATASET_DIR}/kick/*.wav"))
hihat_files = sorted(glob.glob(f"{DATASET_DIR}/hihat/*.wav"))

rows = []
for filepath in kick_files:
    feats = extreu_features(filepath)
    feats['fitxer'] = filepath.split('/')[-1]
    feats['classe'] = 'kick'
    rows.append(feats)

for filepath in hihat_files:
    feats = extreu_features(filepath)
    feats['fitxer'] = filepath.split('/')[-1]
    feats['classe'] = 'hihat'
    rows.append(feats)

df = pd.DataFrame(rows)
df.to_csv('features_percussio.csv', index=False)
print(f"Dataset desat: {df.shape[0]} sons, {df.shape[1]} columnes")
df.head()

In [ ]:
def _test_dataset_final():
    print("Test 3: dataset complet...")
    assert len(df) >= 25, f"❌ Esperava almenys 25 sons, n'hi ha {len(df)}"
    assert set(df['classe'].unique()) == {'kick', 'hihat'}, "❌ Haurien de ser exactament 2 classes"
    assert not df.isnull().values.any(), "❌ Hi ha valors buits al dataset — revisa extreu_features()"
    print(f"   {len(df)} sons totals: {(df['classe']=='kick').sum()} kicks, {(df['classe']=='hihat').sum()} hihats")
    print("✅ Dataset complet i correcte — llest per a la Sessió 12\n")

_test_dataset_final()

## Per provar tu mateix (opcional)

Si vols, prova de calcular `freq_pic()` sobre un dels teus propis sons (gravat amb el micròfon, com vau fer a la Sessió 2) i compara'l amb el centroide espectral de `librosa`. Són la mateixa cosa, o donen valors diferents? Per què?

## Recordatori

El fitxer `features_percussio.csv` que has generat és exactament el que necessitaràs a la Sessió 12 — guarda'l (o assegura't que el notebook el regenera) abans de la propera classe.